# Chapter 3 Practical 02: Cosine, Pearson, and Jaccard Similarity

Learning objectives:
- Compute user-user similarity on co-rated items.
- Compare cosine and Pearson similarity for explicit ratings.
- Use Jaccard similarity for binary interactions.
- Apply minimum-overlap and shrinkage to reduce noisy similarities.

Slide connection: similarity measures, cosine equation, Pearson equation, and sparse overlap.


In [1]:
# Teaching note: Load ratings and movies with a Colab-safe fallback. Local files are used first; GitHub raw CSVs are used when opened directly from GitHub.
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_03_collaborative_filtering/data"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_03_collaborative_filtering/data"

def read_chapter3_csv(filename):
    for data_dir in DATA_DIRS:
        csv_path = data_dir / filename
        if csv_path.exists():
            print(f"Loaded {filename} from {csv_path}")
            return pd.read_csv(csv_path)
    url = f"{GITHUB_DATA_URL}/{filename}"
    print(f"Local file not found. Loading {filename} from GitHub raw URL.")
    return pd.read_csv(url)

ratings = read_chapter3_csv("ratings_chapter3.csv")
movies = read_chapter3_csv("movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
# Pivot interactions into rows = users and columns = items.
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


Loaded ratings_chapter3.csv from data/ratings_chapter3.csv
Loaded movies_chapter3.csv from data/movies_chapter3.csv


title,Blade Runner,Finding Nemo,Independence Day,Jurassic Park,Star Wars,Terminator 2,The Matrix,The Notebook,Titanic,Toy Story
user_id,,,,,,,,,,
Alice,5.0,NaN,4.0,NaN,4.0,NaN,5.0,NaN,NaN,NaN
Bob,NaN,NaN,6.0,4.0,7.0,4.0,7.0,NaN,NaN,NaN
Chris,NaN,NaN,2.0,7.0,3.0,7.0,NaN,NaN,NaN,5.0
Karen,NaN,NaN,NaN,4.0,7.0,3.0,6.0,NaN,NaN,NaN
Lynn,NaN,NaN,2.0,4.0,4.0,6.0,NaN,NaN,NaN,6.0
Nina,NaN,5.0,NaN,4.0,NaN,NaN,NaN,NaN,2.0,5.0
Omar,NaN,2.0,NaN,3.0,NaN,NaN,NaN,5.0,5.0,NaN
Sally,NaN,NaN,7.0,6.0,7.0,3.0,6.0,NaN,NaN,NaN


In [2]:
# Teaching note: Define cosine, Pearson, and Jaccard similarity on shared user histories.
# Keep only co-rated items because user-user similarity should compare shared evidence.
def common_ratings(matrix, user_a, user_b):
    pair = matrix.loc[[user_a, user_b]].dropna(axis=1)
    return pair.loc[user_a], pair.loc[user_b]

# Cosine compares the direction of two rating vectors on shared items.
def cosine_on_overlap(matrix, user_a, user_b):
    a, b = common_ratings(matrix, user_a, user_b)
    if len(a) == 0:
        return np.nan
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return np.nan if denom == 0 else float(np.dot(a, b) / denom)

# Pearson removes each user average before comparing rating patterns.
def pearson_on_overlap(matrix, user_a, user_b):
    a, b = common_ratings(matrix, user_a, user_b)
    if len(a) < 2:
        return np.nan
    if a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

# Jaccard compares overlap between sets of liked items for binary-style feedback.
def jaccard_liked(matrix, user_a, user_b, threshold=5):
    liked_a = set(matrix.columns[matrix.loc[user_a] >= threshold])
    liked_b = set(matrix.columns[matrix.loc[user_b] >= threshold])
    if not liked_a and not liked_b:
        return np.nan
    return len(liked_a & liked_b) / len(liked_a | liked_b)


In [3]:
# Teaching note: Compare one target user with every other user using multiple similarity measures.
rows = []
target = "Karen"
for other in rating_matrix.index.drop(target):
    overlap_count = rating_matrix.loc[[target, other]].notna().all(axis=0).sum()
    rows.append({
        "target_user": target,
        "other_user": other,
        "co_rated_items": int(overlap_count),
        "cosine": cosine_on_overlap(rating_matrix, target, other),
        "pearson": pearson_on_overlap(rating_matrix, target, other),
        "jaccard_liked": jaccard_liked(rating_matrix, target, other),
    })

similarities = pd.DataFrame(rows).sort_values("pearson", ascending=False)
similarities.round(3)


,target_user,other_user,co_rated_items,cosine,pearson,jaccard_liked
1,Karen,Bob,4,0.995,0.949,0.667
6,Karen,Sally,4,0.987,0.843,0.500
3,Karen,Lynn,3,0.874,-0.693,0.000
2,Karen,Chris,3,0.787,-0.971,0.000
0,Karen,Alice,2,0.982,-1.000,0.333
4,Karen,Nina,1,1.000,NaN,0.000
5,Karen,Omar,1,1.000,NaN,0.000


Pearson removes each user's average rating behavior. This matters when one user rates generously and another rates strictly.


In [4]:
# Teaching note: Build a full user-user similarity matrix for Pearson correlation.
def similarity_matrix(metric):
    users = rating_matrix.index
    sim = pd.DataFrame(index=users, columns=users, dtype=float)
    for u in users:
        for v in users:
            sim.loc[u, v] = metric(rating_matrix, u, v) if u != v else 1.0
    return sim

pearson_sim = similarity_matrix(pearson_on_overlap)
pearson_sim.round(2)


user_id,Alice,Bob,Chris,Karen,Lynn,Nina,Omar,Sally
user_id,,,,,,,,
Alice,1.0,0.50,NaN,-1.00,NaN,NaN,NaN,-1.00
Bob,0.5,1.00,-0.91,0.95,-0.54,NaN,NaN,0.66
Chris,NaN,-0.91,1.00,-0.97,0.68,-1.0,NaN,-0.75
Karen,-1.0,0.95,-0.97,1.00,-0.69,NaN,NaN,0.84
Lynn,NaN,-0.54,0.68,-0.69,1.00,1.0,NaN,-0.86
Nina,NaN,NaN,-1.00,NaN,1.00,1.0,-1.0,NaN
Omar,NaN,NaN,NaN,NaN,NaN,-1.0,1.0,NaN
Sally,-1.0,0.66,-0.75,0.84,-0.86,NaN,NaN,1.00


Shrinkage discounts similarities based on very small overlap.


In [5]:
# Teaching note: Apply shrinkage so tiny overlaps produce less confident similarity scores.
# Shrinkage lowers similarity confidence when only a few items overlap.
def shrink_similarity(similarity, overlap_count, alpha=3):
    if pd.isna(similarity):
        return np.nan
    return similarity * overlap_count / (overlap_count + alpha)

similarities["pearson_shrunk"] = similarities.apply(
    lambda row: shrink_similarity(row["pearson"], row["co_rated_items"], alpha=3),
    axis=1,
)
similarities.sort_values("pearson_shrunk", ascending=False).round(3)


,target_user,other_user,co_rated_items,cosine,pearson,jaccard_liked,pearson_shrunk
1,Karen,Bob,4,0.995,0.949,0.667,0.542
6,Karen,Sally,4,0.987,0.843,0.500,0.482
3,Karen,Lynn,3,0.874,-0.693,0.000,-0.347
0,Karen,Alice,2,0.982,-1.000,0.333,-0.400
2,Karen,Chris,3,0.787,-0.971,0.000,-0.485
4,Karen,Nina,1,1.000,NaN,0.000,NaN
5,Karen,Omar,1,1.000,NaN,0.000,NaN


## Challenge Lab

1. Change the liked threshold for Jaccard from `5` to `6`. Which user pairs become less similar?
2. Increase the shrinkage `alpha` from `3` to `10`. Which neighbors lose the most trust?
3. Compare cosine and Pearson for one pair of users. Which metric better handles generous or strict raters?
4. Add a `min_overlap=3` filter before ranking neighbors. Which recommendations become impossible because of sparsity?
